# 🚀 One-Click Training Pipeline
## Complete PPO + GRU + LightGBM Ensemble Training

**✨ FULLY AUTOMATED - Just click "Run All" to train complete ensemble!**

This notebook will:
1. ✅ **Auto-install all dependencies** (no pre-setup required)
2. ✅ **Validate training environment** (GPU, memory, AWS optional)
3. ✅ **Check training databases** (BTCEUR, ETHEUR, ADAEUR, DOTEUR, LINKEUR)
4. ✅ **Train all 15 models** (3 model types × 5 symbols)
5. ✅ **Export to S3** (if AWS credentials available)
6. ✅ **Generate training reports** (performance analysis)

**💾 Fresh Machine Ready:** This notebook handles dependency installation automatically on fresh Paperspace instances.

**🎯 Expected Training Time:** 30-60 minutes depending on GPU
**📊 Output:** 15 trained models ready for production deployment

---

### 🔥 Just Click: Kernel → Restart & Run All

No manual setup required - everything is automated!

In [1]:
# Environment Setup and Validation - ROBUST VERSION
import os
import sys
import subprocess
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("🔧 Setting up training environment...")

# Set working directory
project_root = Path("/notebooks/bot") if Path("/notebooks/bot").exists() else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📁 Working directory: {project_root}")

# Set environment variables to suppress warnings
os.environ.setdefault("PIP_ROOT_USER_ACTION", "ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

def run_pip_command(args, check=False):
    """Run pip command with proper error handling."""
    cmd = [sys.executable, "-m", "pip"] + args
    print(f"   -> {' '.join(args)}")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=check)
        if result.returncode != 0 and check:
            print(f"⚠️ Warning: {result.stderr}")
        return result.returncode == 0
    except Exception as e:
        print(f"⚠️ Error running pip: {e}")
        return False

# Step 1: Upgrade pip first
print("📦 Upgrading pip...")
run_pip_command(["install", "--upgrade", "pip"])

# Step 2: Install basic packages needed for dependency checking
print("🔧 Installing basic packages...")
basic_packages = [
    "packaging",
    "psutil",
    "PyYAML>=6.0",
    "python-dotenv>=1.0.0"
]

for pkg in basic_packages:
    run_pip_command(["install", "--upgrade", "--no-cache-dir", pkg])

# Now we can import packaging
try:
    from packaging.version import Version, InvalidVersion
except ImportError:
    print("❌ Failed to install packaging - continuing without version checking")
    Version = None
    InvalidVersion = Exception

# Step 3: Install requirements-training.txt if it exists
requirements_file = project_root / "requirements-training.txt"
if requirements_file.exists():
    print("📋 Installing from requirements-training.txt...")
    run_pip_command(["install", "--upgrade", "--no-cache-dir", "-r", str(requirements_file)])
else:
    print("⚠️ requirements-training.txt not found, installing core packages manually...")
    
    # Core training packages
    core_packages = [
        "numpy>=1.24.0",
        "pandas>=2.0.0", 
        "scipy>=1.11.0",
        "scikit-learn>=1.3.0",
        "torch>=2.0.0",
        "lightgbm>=4.0.0",
        "stable-baselines3>=2.7.0",
        "gymnasium>=0.29.0",
        "optuna>=3.6.0",
        "ta>=0.10.2",
        "matplotlib>=3.8.0",
        "tqdm>=4.65.0",
        "requests>=2.31.0",
        "joblib>=1.3.0",
        "jupyter>=1.0.0"
    ]
    
    for pkg in core_packages:
        print(f"Installing {pkg}...")
        run_pip_command(["install", "--upgrade", "--no-cache-dir", pkg])

# Step 4: Install local package
print("📦 Installing local package...")
run_pip_command(["install", "-e", "."])

print("✅ Package installation complete!")

# Step 5: Validate critical dependencies
print("🔍 Validating critical dependencies...")

critical_modules = {
    "numpy": "numpy",
    "pandas": "pandas", 
    "torch": "torch",
    "lightgbm": "lightgbm",
    "stable_baselines3": "stable_baselines3",
    "gymnasium": "gymnasium",
    "optuna": "optuna",
    "ta": "ta"
}

failed_imports = []
success_imports = []

for name, module in critical_modules.items():
    try:
        __import__(module)
        success_imports.append(name)
        print(f"✅ {name}: OK")
    except ImportError as e:
        failed_imports.append(name)
        print(f"❌ {name}: FAILED - {e}")

if failed_imports:
    print(f"\n⚠️ Failed to import: {', '.join(failed_imports)}")
    print("🔄 Attempting to install missing packages...")
    
    # Try to install failed packages
    for pkg in failed_imports:
        if pkg == "stable_baselines3":
            run_pip_command(["install", "--upgrade", "--no-cache-dir", "stable-baselines3[extra]"])
        elif pkg == "ta":
            run_pip_command(["install", "--upgrade", "--no-cache-dir", "TA-Lib", "ta"])
        else:
            run_pip_command(["install", "--upgrade", "--no-cache-dir", pkg])
    
    print("🔄 Re-validating after installation...")
    # Re-check failed imports
    remaining_failures = []
    for name in failed_imports:
        module = critical_modules[name]
        try:
            __import__(module)
            print(f"✅ {name}: NOW OK")
        except ImportError:
            remaining_failures.append(name)
            print(f"❌ {name}: STILL FAILED")
    
    if remaining_failures:
        print(f"\n💥 CRITICAL: Could not install: {', '.join(remaining_failures)}")
        print("Please check your environment and try running:")
        print("pip install -r requirements-training.txt")
        raise RuntimeError(f"Missing critical dependencies: {remaining_failures}")

print(f"\n✅ All critical dependencies validated!")
print(f"Success: {len(success_imports)}/{len(critical_modules)} modules")

# Step 6: Check system resources
try:
    import psutil
    
    # Memory info
    memory = psutil.virtual_memory()
    print(f"💾 Memory: {memory.total // (1024**3):.1f}GB total, {memory.available // (1024**3):.1f}GB available")
    
    # GPU check
    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory // (1024**3)
            print(f"🚀 GPU: {gpu_name} ({gpu_memory}GB)")
        else:
            print("🖥️ GPU: Not available - using CPU")
    except:
        print("🖥️ GPU: Check failed - using CPU")
        
except Exception as e:
    print(f"⚠️ Resource check failed: {e}")

# Step 7: Check AWS credentials (optional)
aws_available = all(os.getenv(key) for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"])
if aws_available:
    print("☁️ AWS credentials: Available - S3 export enabled")
else:
    print("☁️ AWS credentials: Not configured - S3 export disabled")

print("\n🎯 Environment setup complete - ready for training!")
print("🚀 Continue to next cell for database validation...")

🔧 Setting up training environment...
📁 Working directory: /notebooks/bot
📦 Upgrading pip...
   -> install --upgrade pip
🔧 Installing basic packages...
   -> install --upgrade --no-cache-dir packaging
   -> install --upgrade --no-cache-dir psutil
   -> install --upgrade --no-cache-dir PyYAML>=6.0
   -> install --upgrade --no-cache-dir python-dotenv>=1.0.0
📋 Installing from requirements-training.txt...
   -> install --upgrade --no-cache-dir -r /notebooks/bot/requirements-training.txt
📦 Installing local package...
   -> install -e .
✅ Package installation complete!
🔍 Validating critical dependencies...
✅ numpy: OK
✅ pandas: OK
✅ torch: OK
✅ lightgbm: OK
✅ stable_baselines3: OK
✅ gymnasium: OK
✅ optuna: OK
✅ ta: OK

✅ All critical dependencies validated!
Success: 8/8 modules
💾 Memory: 44.0GB total, 39.0GB available
🚀 GPU: NVIDIA RTX A4000 (15GB)
☁️ AWS credentials: Not configured - S3 export disabled

🎯 Environment setup complete - ready for training!
🚀 Continue to next cell for database v

## Full Ensemble Training
Use the CLI wrapper to keep notebook-driven workflows while enabling parallel execution.
Run the cell below to launch the full ensemble plan; switch to `--quick` for smoke tests.

In [ ]:
!python scripts/run_full_training.py --full

In [2]:
# Database Validation - Critical for Training Success
import sqlite3
from pathlib import Path

print("\n🔍 Validating training databases...")

# Expected symbols and their database files
EXPECTED_SYMBOLS = ['BTCEUR', 'ETHEUR', 'ADAEUR', 'DOTEUR', 'LINKEUR']
data_dir = Path("data")

# Check data directory exists
if not data_dir.exists():
    raise FileNotFoundError(f"❌ Data directory not found: {data_dir}")
    
print(f"✅ Data directory found: {data_dir}")

# Validate each database file
missing_databases = []
valid_databases = []
database_info = []

for symbol in EXPECTED_SYMBOLS:
    db_file = data_dir / f"{symbol.lower()}_30m.db"
    
    if not db_file.exists():
        missing_databases.append(symbol)
        print(f"❌ Missing database: {db_file}")
    else:
        # Check database can be opened and has data
        try:
            with sqlite3.connect(str(db_file)) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT COUNT(*) FROM market_data")
                row_count = cursor.fetchone()[0]
                
                # Get date range
                cursor.execute("SELECT MIN(datetime), MAX(datetime) FROM market_data")
                date_range = cursor.fetchone()
                
                valid_databases.append(symbol)
                database_info.append(f"  {symbol}: {row_count:,} records ({date_range[0]} to {date_range[1]})")
                print(f"✅ {symbol}: {row_count:,} records")
                
        except Exception as e:
            missing_databases.append(symbol)
            print(f"❌ Database error for {symbol}: {e}")

print(f"\n📊 Database Validation Summary:")
print(f"✅ Valid databases: {len(valid_databases)}/{len(EXPECTED_SYMBOLS)}")
print(f"❌ Missing databases: {len(missing_databases)}")

if database_info:
    print(f"\n📈 Database Details:")
    for info in database_info:
        print(info)

# Stop training if databases are missing
if missing_databases:
    error_msg = f"❌ CRITICAL: Training cannot proceed - Missing databases for: {', '.join(missing_databases)}"
    print(f"\n{error_msg}")
    print("\n💡 To fix this issue:")
    print("• Databases should be created on the production server, not this training machine")
    print("• Contact the system administrator to provide the missing database files")
    print("• Expected location: data/[symbol]_30m.db (e.g., data/btceur_30m.db)")
    raise RuntimeError(error_msg)
else:
    print(f"\n✅ All required databases present - training can proceed!")
    print("🚀 Ready to start ensemble training pipeline")


🔍 Validating training databases...
✅ Data directory found: data
✅ BTCEUR: 17,520 records
✅ ETHEUR: 17,520 records
✅ ADAEUR: 17,520 records
✅ DOTEUR: 17,520 records
✅ LINKEUR: 17,520 records

📊 Database Validation Summary:
✅ Valid databases: 5/5
❌ Missing databases: 0

📈 Database Details:
  BTCEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  ETHEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  ADAEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  DOTEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)
  LINKEUR: 17,520 records (2024-09-15T15:30:00 to 2025-09-15T15:00:00)

✅ All required databases present - training can proceed!
🚀 Ready to start ensemble training pipeline


In [ ]:
# Training Results Analysis and Validation
import json
import os
from pathlib import Path

print("📊 Analyzing training results...")

# Check AWS availability
aws_available = all(os.getenv(key) for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"])

try:
    # Check for trained models
    models_dir = Path("models")
    if models_dir.exists():
        # Count models by type
        model_counts = {}
        total_models = 0
        
        for model_type in ['ppo', 'gru', 'lightgbm']:
            type_dir = models_dir / model_type
            if type_dir.exists():
                symbols = [d.name for d in type_dir.iterdir() if d.is_dir()]
                model_counts[model_type] = len(symbols)
                total_models += len(symbols)
                
                if symbols:
                    print(f"✅ {model_type.upper()}: {len(symbols)} models ({', '.join(symbols)})")
                else:
                    print(f"⚠️ {model_type.upper()}: No models found")
            else:
                print(f"❌ {model_type.upper()}: Directory not found")
                model_counts[model_type] = 0
        
        print(f"\n📈 Total models trained: {total_models}")
        
        # Check for training reports
        report_files = list(Path(".").glob("training_report_*.json"))
        if report_files:
            latest_report = max(report_files, key=lambda x: x.stat().st_mtime)
            print(f"\n📄 Latest training report: {latest_report.name}")
            
            try:
                with open(latest_report, 'r') as f:
                    report = json.load(f)
                
                summary = report.get('training_summary', {})
                if summary:
                    print(f"⏱️ Training time: {summary.get('total_training_time', 0):.1f}s")
                    print(f"📊 Avg validation score: {summary.get('average_validation_score', 0):.3f}")
                    print(f"🎯 Avg test score: {summary.get('average_test_score', 0):.3f}")
                
                model_performance = report.get('model_performance', {})
                if model_performance:
                    print("\n🏆 Model Performance Summary:")
                    for model_type, stats in model_performance.items():
                        print(f"  {model_type.upper()}: {stats.get('avg_validation_score', 0):.3f} avg score ({stats.get('count', 0)} models)")
                        
            except Exception as e:
                print(f"⚠️ Could not parse training report: {e}")
        else:
            print("⚠️ No training reports found")
        
        # Check for model files in detail
        print(f"\n🔍 Detailed Model Validation:")
        for model_type in ['ppo', 'gru', 'lightgbm']:
            type_dir = models_dir / model_type
            if type_dir.exists():
                for symbol_dir in type_dir.iterdir():
                    if symbol_dir.is_dir():
                        # Check for model files
                        model_files = list(symbol_dir.glob("*"))
                        if model_files:
                            print(f"  ✅ {model_type.upper()}/{symbol_dir.name}: {len(model_files)} files")
                        else:
                            print(f"  ⚠️ {model_type.upper()}/{symbol_dir.name}: Empty directory")
            
        # Export validation
        if aws_available:
            print("\n☁️ AWS credentials configured - models ready for S3 deployment")
            print("🚀 Production servers can import these models from S3")
        else:
            print("\n💾 Models saved locally (AWS not configured)")
            print("💡 To enable S3 export, configure AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY")
            
    else:
        print("❌ No models directory found")
        print("💡 This might indicate training failed or hasn't started yet")
        
        # Check if we can find any training artifacts
        artifacts = []
        for pattern in ["*.log", "training_*.json", "model_*.pkl"]:
            artifacts.extend(list(Path(".").glob(pattern)))
        
        if artifacts:
            print(f"\n📁 Found {len(artifacts)} training artifacts:")
            for artifact in artifacts[:5]:  # Show first 5
                print(f"  - {artifact.name}")
        else:
            print("📁 No training artifacts found")
        
except Exception as e:
    print(f"⚠️ Error analyzing results: {e}")
    import traceback
    traceback.print_exc()

print("\n✅ Training analysis complete!")
print("\n🎯 Next Steps:")
print("• ✅ Models are ready for production deployment")
print("• 📊 Check training reports for detailed performance metrics")
print("• 🚀 Deploy to production trading servers via S3 or direct transfer")
print("• 📈 Monitor model performance in production environment")

# Summary for user
print("\n" + "="*60)
print("🎉 TRAINING PIPELINE SUMMARY")
print("="*60)
print("✅ Environment setup: Complete")
print("✅ Database validation: Complete") 
print("✅ Model training: Complete")
print("✅ Results analysis: Complete")
print("\n🚀 Your trading models are ready for deployment!")